# Demo Generative AI LLM bot

There are many Generative AI models and systems. We could try to get a connection to one of the big AI company AIs, but they may require payment for tokens (e.g., Claude requires a minimum of $5 to [get an API key from here](https://platform.claude.com/dashboard)).

Instead we can download models and run them ourselves. You can search for downloadable models from [Hugging Face](https://huggingface.co/).

We will use a small Large Language Model (LLM) called [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct).

## Download and install SmolLM2-135M-Instruct

In [2]:
!pip install transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"

# see website for model: https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct

# possible alternatives (some may be too large to run on your instance)
# checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
# checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
# checkpoint = "HuggingFaceTB/SmolLM3-3B"

# device = "cuda" # for GPU usage or "cpu" for CPU usage
device = "cpu"

## Test LLM (talk like a pirate)

In [5]:
# Start new LLM session (I think)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

# Test transforming a phrase from talk to a pirate
user_message = "It's a nice day outside, I think I'll enjoy the sun!"
print("user message to test: " + user_message)
print()

messages = [{"role": "system", "content": "You are a pirate. Respond with the most piratey words and phrases."},
            {"role": "user", "content": user_message}]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print("Applying user text:")
print()
print(input_text)
print("----------")

inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
full_output_text = tokenizer.decode(outputs[0])

print("full chat text:")
print()
print(full_output_text)
print("----------")

# get the output
output_response = output_response = full_output_text.split("\n<|im_start|>assistant\n")[1].replace("<|im_end|>", "")
print("model output:")
print()
print(output_response)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


user message to test: It's a nice day outside, I think I'll enjoy the sun!

Applying user text:

<|im_start|>system
You are a pirate. Respond with the most piratey words and phrases.<|im_end|>
<|im_start|>user
It's a nice day outside, I think I'll enjoy the sun!<|im_end|>

----------
full chat text:

<|im_start|>system
You are a pirate. Respond with the most piratey words and phrases.<|im_end|>
<|im_start|>user
It's a nice day outside, I think I'll enjoy the sun!<|im_end|>
<|im_start|>assistant
A fine day indeed! I'll be sailin' the seven seas, plunderin' the riches of the landlubbers!<|im_end|>
----------
model output:

A fine day indeed! I'll be sailin' the seven seas, plunderin' the riches of the landlubbers!


## Demo Disagreement Bot
When a user tags our bot, this code takes whatever the user says and then uses an LLM to make a reply by disagreeing with the user.

This is a potentially harmful bot. While our bot here only replies to people who message our bot, real bots on Bluesky have been found before that spam popular posts with disagreements. This is also similar to how Russia has tried to disrupt US politics and society by trying to get Americans upset at each other over politics or even pop culture (e.g., [Star Wars movies](https://www.wired.com/story/star-wars-russian-trolls-study/)).

Also related to this is the joke "[disc horse bot](https://bsky.app/profile/horsedisc.bsky.social)" (sounds like the word "discourse," meaning "what are people arguing about currently"). The "disk horse bot" automatically creates invented controversial statements like "Eating glue is the weaponization of democracy."

### Log into BlueSky

In [6]:
# Load some code called "Client" from the "atproto" library that will help us work with Bluesky
from atproto import Client

In [7]:
# Login to Bluesky
%run bluesky_keys.py

client = Client(base_url="https://bsky.social")
client.login(handle, password)

ProfileViewDetailed(did='did:plc:cqnztqpi453mwq2yzh35voz4', handle='info103-teacher.bsky.social', associated=ProfileAssociated(chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated', activitySubscription={'allowSubscriptions': 'followers'}), avatar=None, banner=None, created_at='2024-04-09T19:36:16.721Z', description=None, display_name='', followers_count=5, follows_count=0, indexed_at='2026-03-30T16:18:32.257Z', joined_via_starter_pack=None, labels=[Label(cts='2024-04-09T19:36:16.976Z', src='did:plc:cqnztqpi453mwq2yzh35voz4', uri='at://did:plc:cqnztqpi453mwq2yzh35voz4/app.bsky.actor.profile/self', val='bot', cid='bafyreiduyy3izxbk5zhkxomk7cz44ibalqdid5e6ipq2yo2jn6k3vbw3au', exp=None, neg=None, sig=None, ver=None, py_type='com.atproto.label.defs#label')], pinned_post=None, posts_count=103, viewer=ViewerState(blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None, muted=False

### Get My Latest Mention

In [8]:
# look up my Bluesky handle
my_handle = client.me.handle

print("my handle is:" + my_handle)

my handle is:info103-teacher.bsky.social


In [9]:
# search posts that mention my handle
mentions = client.app.bsky.feed.search_posts({
    'q': my_handle, 
    'mentions': my_handle
}).posts

if(len(mentions) < 1):
    print("Error: Could not find matching mentions!")
    
latest_mention = mentions[0]

# display the text of the post, so we can see what we found
display("latest post text: " + str(latest_mention.record.text))

'latest post text: Hi @info103-teacher.bsky.social, broccoli is best when it is fried with garlic!'

### Get response that disagrees

In [10]:
# Start new LLM session (I think)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

# get rid of the handle in the incoming message
message_to_respond_to = latest_mention.record.text.replace("@info103-teacher.bsky.social", "")

# Test transforming a phrase from talk to a pirate
messages = [
    {"role": "system", "content": 'You are not an assistant but the opposite side of a debate. You disagree with whatever the statement says statement, no exceptions. Say the opposite is true.'},
    {"role": "user", "content": message_to_respond_to}
]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print("Applying user text:")
print(input_text)
print()
print("----------")

inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=100, temperature=0.2, top_p=0.9, do_sample=True)
full_output_text = tokenizer.decode(outputs[0])

print("full chat text:")
print(full_output_text)
print()
print("----------")

# get the output
output_response = output_response = full_output_text.split("\n<|im_start|>assistant\n")[1]
print("model output:")
print()
print(output_response)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Applying user text:
<|im_start|>system
You are not an assistant but the opposite side of a debate. You disagree with whatever the statement says statement, no exceptions. Say the opposite is true.<|im_end|>
<|im_start|>user
Hi , broccoli is best when it is fried with garlic!<|im_end|>


----------
full chat text:
<|im_start|>system
You are not an assistant but the opposite side of a debate. You disagree with whatever the statement says statement, no exceptions. Say the opposite is true.<|im_end|>
<|im_start|>user
Hi , broccoli is best when it is fried with garlic!<|im_end|>
<|im_start|>assistant
I'm glad you're happy with the broccoli you're enjoying. I'm not a food critic, but I can tell you that when it's fried with garlic, it's not the best option. Garlic can be quite bitter, and it can also be quite strong, which can make it difficult to digest.

If you're looking for a healthier option, I would recommend grilling the broccoli with a bit of olive oil and a bit of garlic, or even us

### Respond to the post that mentioned me

In [11]:
# make sure response isn't too long

response = output_response[:299]

client.send_post(
    response, 
    reply_to={'root': {'uri': latest_mention.uri, 'cid': latest_mention.cid}, 
              'parent': {'uri': latest_mention.uri, 'cid': latest_mention.cid}}
)

CreateRecordResponse(uri='at://did:plc:cqnztqpi453mwq2yzh35voz4/app.bsky.feed.post/3mmzdckbuxg2d', cid='bafyreigfoov6fpt4uregogsgl7p765pd2sqmhi66wv7jrsguob72x5ccwq')